In [1]:
import os
import sys
import random
import optuna
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from pandas.api.types import is_numeric_dtype

sys.path.append("..")
from feature_engineering import time_based_train_test_split, get_return, round_to_step, get_mask, add_week_column
from etl_pipeline_local import get_over_15_table

sys.path.append("../..")
from strategies.utils import analyze_strategies, get_best_strategies

from utils import trial_key, run_one_study
pd.set_option('display.max_columns', None)
optuna.logging.set_verbosity(optuna.logging.INFO)

load_dotenv()

True

In [2]:
# Bet setup
target_col = "over_15"
odds_col: str = "over_15_odds"

# Load data
df_loaded = get_over_15_table()
df_loaded = df_loaded.sort_values('time').reset_index(drop=True)
df = df_loaded.copy()

# Feature Engineering
df[odds_col] = 1.2
df["return"] = df.apply(lambda x: get_return(strategy=x[target_col], odds=x[odds_col]), axis=1).copy()
df["week"] = add_week_column(df)
# df = df[df["underOver_quote_currentU"] <= 1.7]

# Train/test split
df_train, df_test = time_based_train_test_split(df=df, time_col="time", train_frac=0.8)

In [9]:
objective_hyperparameters = {
    "n_trials": 500, # 500 - 1000
    "n_studies": 1, # 50
    "min_obs": 100, 
    "lambda": 1, # 0.5, 1, 2
    "tpe_sampler": 
    {
        "n_startup_trials": 10, # 100, # 10 - 30 per search space semplici, 50 100+ per seearch space con molti minimi
        "n_ei_candidates": 24, #128 # 24 default, problemi difficili: 64 o 128
    }
}

In [10]:
goalNoGoal_features = list(df.filter(like="goalNoGoal").columns)
underOver_features = list(df.filter(like="underOver").columns)
evaluation_features = list(df.filter(like="evaluation").columns)
features = underOver_features # goalNoGoal_features + underOver_features + evaluation_features

In [17]:
def _get_step(feature):
    q1 = df[[feature]].describe().loc["25%"].values[0]
    q3 = df[[feature]].describe().loc["75%"].values[0]

    diff = q3 - q1
    
    if df[feature].nunique() <= 20:
        return None

    if 0 < diff <= 1:
        step = 0.2

    elif 1 < diff <= 5:
        step = 1

    elif 5 < diff <= 100:
        step = 5
    elif 100 < diff <= 500:
        step = 100
    elif diff >= 500:
        step = 500
    else: 
        raise Exception
    return step

def get_feature_bins_map(df):
    feature_bins_map = dict()

    for feat in features:
        if is_numeric_dtype(df[feat]):
            feature_bins_map[feat] = _get_step(feat)
        else:
            feature_bins_map[feat] = None
    return feature_bins_map


In [ ]:
from itertools import combinations

def objective_features(trial, min_obs:int, _lambda: float, feature_bins_map: dict, df_binned: pd.DataFrame):
    mask = pd.Series(True, index=df_binned.index)
    params_dict_item = dict()

    for feat, step in feature_bins_map.items():

        s = df_binned[feat]
        non_null = s.dropna()

        if non_null.empty:
            continue

        # ---------------------------------
        # CASO 1: variabile categorica
        # ---------------------------------
        if not step:
            # ordine stabile e deterministico
            unique_vals = sorted(map(str, non_null.unique()))

            categorical_set = [
                list(c)
                for r in range(1, len(unique_vals) + 1)
                for c in combinations(unique_vals, r)
            ]

            idx = trial.suggest_categorical(
                f"{feat}_cat_idx",
                list(range(len(categorical_set)))
            )

        
            categorical_feat = categorical_set[idx]
            # trial.set_user_attr(f"{feat}_cat", categorical_feat)
            params_dict_item[f"{feat}_cat"] = categorical_feat

            # use_feat = trial.suggest_categorical(f"use_{feat}", [True, False])
            # params_dict_item[f"use_{feat}"] = use_feat


        # ---------------------------------
        # CASO 2: numerica continua/intera
        # ---------------------------------
        else:
            # TODO: se la feature ha valori nulli, introduco include_missing feature, valutare rimozione  
            if s.isna().sum() > 0:
                include_missing = trial.suggest_categorical(
                    f"{feat}_include_missing",
                    [True, False]
                )
                # trial.set_user_attr(f"{feat}_include_missing", include_missing)
                params_dict_item[f"{feat}_include_missing"] = include_missing

            # TODO: vincolo per >=, <=, <,>, da studiare
            use_min = trial.suggest_categorical(f"{feat}_use_min", [True, False])
            use_max = trial.suggest_categorical(f"{feat}_use_max", [True, False])

            # trial.set_user_attr(f"{feat}_use_min", use_min)
            # trial.set_user_attr(f"{feat}_use_max", use_max)
            params_dict_item[f"{feat}_use_min"] = use_min
            params_dict_item[f"{feat}_use_max"] = use_max

            # use_feat = trial.suggest_categorical(f"use_{feat}", [True, False])
            # params_dict_item[f"use_{feat}"] = use_feat
            

            # TODO: questo dovrebbe forzare l'utilizzo della feature, da capire se forzarlo è utile o meno
            # # almeno un vincolo deve essere attivo
            # if not use_min and not use_max:
            #     raise optuna.TrialPruned()
            
            if pd.api.types.is_integer_dtype(s):
                feat_min = trial.suggest_int(
                    name=f"{feat}_min",
                    low=int(non_null.min()),
                    high=int(non_null.max()) - step, # considero upper bound meno step
                    step=step
                )
                feat_max = trial.suggest_int(
                    name=f"{feat}_max",
                    low=int(non_null.min()), # feat_min + step, # considero lower bound più step
                    high=int(non_null.max()), 
                    step=step
                )
                # trial.set_user_attr(f"{feat}_min", feat_min)
                # trial.set_user_attr(f"{feat}_max", feat_max)
                if feat_min >= feat_max:
                    raise optuna.TrialPruned()
                params_dict_item[f"{feat}_min"] = feat_min
                params_dict_item[f"{feat}_max"] = feat_max
            else:
                feat_min = trial.suggest_float(
                    name=f"{feat}_min",
                    low=float(non_null.min()),
                    high=float(non_null.max()) - step, # considero upper bound meno step
                    step=step
                )
                feat_max = trial.suggest_float(
                    name=f"{feat}_max",
                    low=float(non_null.min()), # feat_min + step, # considero lower bound più step
                    high=float(non_null.max()),
                    step=step
                )
                # trial.set_user_attr(f"{feat}_min", feat_min)
                # trial.set_user_attr(f"{feat}_max", feat_max)
                if feat_min >= feat_max:
                    raise optuna.TrialPruned()
                params_dict_item[f"{feat}_min"] = feat_min
                params_dict_item[f"{feat}_max"] = feat_max

        trial.set_user_attr("params_dict", params_dict_item)

        # costruzione maschera feature
        feat_mask = pd.Series(True, index=s.index)

        if not step:
            # Categorical filtering
            feat_mask &= s.isin(categorical_feat)
        else:
            # Numerical filtering
            if use_min:
                feat_mask &= s >= feat_min

            if use_max:
                feat_mask &= s <= feat_max

            # Filtering for feature with nan
            if s.isna().sum() > 0:
                if include_missing:
                    feat_mask = feat_mask | s.isna()
                else:
                    feat_mask = feat_mask & s.notna()

        mask &= feat_mask

    selected = df_binned.loc[mask]

    # if len(selected) <= 0:
    #     return -1e9
    
    mean_weekly_return = selected.groupby("week")["return"].sum().mean() # massimizzo ritorno settimanale
    std_weekly_return = selected.groupby("week")["return"].sum().std(ddof=0) # minimizzo volatilità ritorno settimanale


    if len(selected) < min_obs:
        return -1e9
    
    score = mean_weekly_return - _lambda * std_weekly_return 

    return float(score)

In [13]:
feature_rank = dict()

for key, value in feature_bins_map.items():
    feature_bins_map_item = {key: value}

    
    # Create the binned dataframe
    df_train_binned = df_train.copy()

    for feat, step in feature_bins_map_item.items():
        df_train_binned[feat] = [round_to_step(x, step) for x in df_train_binned[feat]]

        if isinstance(step, int):
            df_train_binned[feat] = df_train_binned[feat].astype("Int64")

    # print("SEARCH SPACE\n")
    # for feat in feature_bins_map_item.keys():
    #     space = df_train_binned[feat].sort_values(ascending=True).unique()
    #     print(f"Feature: {feat}")
    #     print(f"Search space: {space}")
    #     print("\n")

    # Generate random seeds
    seeds = [random.randint(0, 2**32 - 1) for _ in range(objective_hyperparameters["n_studies"])]
    studies = [
        run_one_study(
            seed=s, 
            objective_hyperparameters=objective_hyperparameters, 
            objective=objective_features,
            feature_bins_map=feature_bins_map_item, 
            df_binned=df_train_binned
            ) 
            for s in seeds]

    all_trials = []
    for study in studies:
        all_trials.extend(
            [t for t in study.trials if t.value is not None and t.state.name == "COMPLETE"]
        )

    best_trial = sorted(all_trials, key=lambda t: -t.value)[0]

    feature_rank[feat] = best_trial.value

[I 2026-04-10 16:03:36,952] A new study created in memory with name: no-name-0d40b171-12ac-4304-8f50-c603a3dffd3e
[I 2026-04-10 16:03:36,961] Trial 0 finished with value: -33.938396101312534 and parameters: {'underOver_chance_under05HT_use_min': False, 'underOver_chance_under05HT_use_max': True, 'use_underOver_chance_under05HT': True, 'underOver_chance_under05HT_min': 45, 'underOver_chance_under05HT_max': 50}. Best is trial 0 with value: -33.938396101312534.
[I 2026-04-10 16:03:36,968] Trial 1 finished with value: -27.693924807503123 and parameters: {'underOver_chance_under05HT_use_min': True, 'underOver_chance_under05HT_use_max': False, 'use_underOver_chance_under05HT': True, 'underOver_chance_under05HT_min': 35, 'underOver_chance_under05HT_max': 50}. Best is trial 1 with value: -27.693924807503123.
[I 2026-04-10 16:03:36,976] Trial 2 finished with value: -34.5278155109082 and parameters: {'underOver_chance_under05HT_use_min': True, 'underOver_chance_under05HT_use_max': False, 'use_un

In [16]:
dict(sorted(feature_rank.items(), key=lambda x: -x[1]))

{'underOver_quote_initialU': -0.030436633405484748,
 'underOver_flashback_over25': -0.04064165761998373,
 'underOver_flashback_under35': -0.04517967689146396,
 'underOver_flashback_over35': -0.04517967689146396,
 'underOver_flashback_under05HT': -0.05940290803742698,
 'underOver_flashback_over05HT': -0.05940290803742698,
 'underOver_bookkeeping_u': -0.07972665122269973,
 'underOver_bookkeeping_o': -0.07972665122269973,
 'underOver_flashback_over15': -0.19331262919989922,
 'underOver_chance_under25': -0.23002188310893334,
 'underOver_chance_over25': -0.23002188310893334,
 'underOver_chance_under45': -0.4078911199379075,
 'underOver_chance_over45': -0.4078911199379075,
 'underOver_chance_under15': -0.5078969433476378,
 'underOver_chance_under15HT': -0.524527223342314,
 'underOver_chance_over15': -0.5458621386928076,
 'underOver_quote_realU': -0.6387070037928365,
 'underOver_chance_over15HT': -0.6841500171254202,
 'underOver_chance_under35': -0.6999042413057153,
 'underOver_chance_over35'